# Discovering registered artists

plotlet's plot types live in a registry. Core artists register on `import plotlet`; 
extensions register lazily when you `import plotlet.extensions.<name>`; user-defined 
artists register via `pt.add_artist(...)`. This notebook shows `pt.artist_table()`, 
the helper for inspecting that registry.

In [1]:
import plotlet as pt


## Quick scan — `artist_table()`

The default view: name / origin / layer / coord_systems. 
Pretty-prints in the REPL and renders as an HTML table in Jupyter.

In [2]:
pt.artist_table()

name,origin,layer,coord_systems
annotate,core,foreground,-
annotation_strip,core,data,{Circular}
area,core,data,-
axhline,core,foreground,-
axhspan,core,background,-
axline,core,foreground,-
axvline,core,foreground,-
axvspan,core,background,-
bar,core,data,-
boxplot,core,data,-


## All columns

Pass `columns="all"` to surface every informative `ArtistSpec` field. The column 
set is derived live from the dataclass, so new fields appear automatically without 
touching this helper. Required plumbing callables (`record`, `draw`) and lambda-defaulted 
fields (`xdomain`, `ydomain`) are skipped — they'd be `"fn"` on every row.

In [3]:
pt.artist_table(columns='all')

name,origin,xdomain_log,ydomain_log,layer,uses_color_cycle,default_color,accepts_data_positional,legend_entries,legend_gradient,data_attrs,flips_y_axis,tight_domain,force_zero_x,force_zero_y,axis_order,frame_defaults,crosses_sectors,coord_systems
annotate,core,-,-,foreground,False,#222222,False,-,-,fn,-,False,False,False,-,-,False,-
annotation_strip,core,-,-,data,False,-,True,fn,fn,-,-,True,False,False,-,fn,False,{Circular}
area,core,-,-,data,True,-,True,fn,-,fn,-,False,False,False,-,-,False,-
axhline,core,-,-,foreground,False,#000000,False,fn,-,fn,-,False,False,False,-,-,False,-
axhspan,core,-,-,background,False,#000000,True,fn,-,fn,-,False,False,False,-,-,False,-
axline,core,-,-,foreground,False,#000000,False,fn,-,fn,-,False,False,False,-,-,False,-
axvline,core,-,-,foreground,False,#000000,False,fn,-,fn,-,False,False,False,-,-,False,-
axvspan,core,-,-,background,False,#000000,True,fn,-,fn,-,False,False,False,-,-,False,-
bar,core,-,-,data,True,-,True,fn,-,fn,-,False,fn,fn,-,-,False,-
boxplot,core,-,-,data,True,-,True,fn,-,-,-,False,False,False,-,-,False,-


## Custom column pick

Pass a list to choose columns explicitly, in the order you want them.

In [4]:
pt.artist_table(columns=['name', 'coord_systems', 'crosses_sectors'])

name,coord_systems,crosses_sectors
annotate,-,False
annotation_strip,{Circular},False
area,-,False
axhline,-,False
axhspan,-,False
axline,-,False
axvline,-,False
axvspan,-,False
bar,-,False
boxplot,-,False


## Dynamic — extensions show up on import

Extension artists live in the separate `plotlet-extensions` package and don't
register with core on their own. Import from the package and its artists appear
in the table on the next call, tagged `extension`.

In [5]:
import plotlet.extensions.sankey   # importing the package registers its artists

[r['name'] for r in pt.artist_table() if r['origin'] == 'extension']

['alluvial',
 'barh',
 'biplot_label',
 'bland_altman',
 'boxen',
 'bubble_grid',
 'bump',
 'calendar_heatmap',
 'calibration',
 'cleveland_dot',
 'confusion_matrix',
 'crossbar',
 'diverging_bar',
 'dumbbell',
 'eventplot',
 'forest',
 'funnel_plot',
 'gene_arrow',
 'horizon',
 'km',
 'loess',
 'lollipop',
 'ma',
 'manhattan',
 'mosaic',
 'parallel_coords',
 'percentile_band',
 'pr',
 'pyramid',
 'raincloud',
 'roc',
 'sales_funnel',
 'sankey',
 'significance_brackets',
 'slope_chart',
 'split_violin',
 'text_label',
 'upset',
 'volcano',
 'waterfall']

## Dynamic — user-added artists

Anything registered via `pt.add_artist(...)` from a non-`plotlet.*` module is 
tagged `user`. Useful for sanity-checking your own extensions.

In [6]:
from plotlet.registry import ArtistSpec

pt.add_artist(ArtistSpec(
    name='my_artist',
    record=lambda *args, **kw: {'type': 'my_artist', 'opts': kw},
    draw=lambda a, ctx: '',
))

[r for r in pt.artist_table() if r['origin'] == 'user']

[{'origin': 'user',
  'name': 'my_artist',
  'record': <function __main__.<lambda>(*args, **kw)>,
  'draw': <function __main__.<lambda>(a, ctx)>,
  'xdomain': <function plotlet.registry.ArtistSpec.<lambda>(a)>,
  'ydomain': <function plotlet.registry.ArtistSpec.<lambda>(a)>,
  'xdomain_log': None,
  'ydomain_log': None,
  'layer': 'data',
  'uses_color_cycle': True,
  'default_color': None,
  'accepts_data_positional': True,
  'legend_entries': None,
  'legend_gradient': None,
  'data_attrs': None,
  'flips_y_axis': None,
  'tight_domain': False,
  'force_zero_x': False,
  'force_zero_y': False,
  'axis_order': None,
  'frame_defaults': None,
  'crosses_sectors': False,
  'coord_systems': set()}]

## Programmatic filtering

The table iterates as a list of dicts carrying the **full** field set 
(regardless of which columns are displayed). Use comprehensions to filter:

In [7]:
# all foreground-layer artists
[r['name'] for r in pt.artist_table() if r['layer'] == 'foreground']

['annotate',
 'axhline',
 'axline',
 'axvline',
 'biplot_label',
 'crossbar',
 'rug',
 'significance_brackets',
 'text',
 'text_label']

In [8]:
# all artists that draw correctly under non-affine coords (CircularCoordinate, ...)
[r['name'] for r in pt.artist_table() if r['coord_systems']]

['annotation_strip', 'chord_links', 'chord_ribbon', 'numeric_bar']